# Notebook 02 — Detection + Team Assignment

**Day 2 goal:** Players are consistently assigned to two teams on a tactical-camera clip.

**What this notebook validates:**
- `FootballDetector` — wraps YOLO, handles frame-skip cache, clean API
- `TeamAssigner`    — K-Means on jersey HSV colour, separates teams reliably

**No tracking yet.** Track IDs come in Day 3 (ByteTrack). Today we just need colours right.

## Cell 1 — Imports & Setup

In [ ]:
import sys, io
if sys.platform == 'win32':
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
        sys.stderr.reconfigure(encoding='utf-8', errors='replace')
    except AttributeError:
        pass

import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

import cv2
import numpy as np
import time
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Image as IPImage

from gaffer import config
from gaffer.detection.detector import FootballDetector, Detection
from gaffer.detection.team_assigner import TeamAssigner

def show(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    buf.seek(0)
    display(IPImage(data=buf.read()))
    plt.close(fig)

ROOT      = Path('..').resolve()
CLIPS_DIR = ROOT / 'data' / 'test_clips'
CLIP      = CLIPS_DIR / 'tactical_playlist_1.mp4'

assert CLIP.exists(), f'Clip not found: {CLIP}'
print(f'Clip: {CLIP.name}  ({CLIP.stat().st_size/1e6:.1f} MB)')
print('Imports OK')

## Cell 2 — Video Metadata

In [ ]:
cap = cv2.VideoCapture(str(CLIP))
W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS      = cap.get(cv2.CAP_PROP_FPS)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f'Resolution : {W}x{H}')
print(f'FPS        : {FPS}')
print(f'Frames     : {N_FRAMES}  ({N_FRAMES/FPS:.0f}s)')

def read_frame(idx):
    cap = cv2.VideoCapture(str(CLIP))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, f = cap.read()
    cap.release()
    if not ret: raise RuntimeError(f'Cannot read frame {idx}')
    return f

## Cell 3 — Initialize FootballDetector

**Design decisions in FootballDetector:**
- Falls back to `yolo11n.pt` if the fine-tuned model doesn't exist yet
- Auto-detects whether it's a COCO or football model from class names
- Warm-up on init so first real inference isn't an outlier
- `detect(frame, frame_idx)` — the single public method; caches for skip-N frames

In [ ]:
detector = FootballDetector(
    conf           = config.DETECTION_CONF_THRESHOLD,  # 0.25 (Day 1 finding)
    imgsz          = config.DETECTION_IMG_SIZE,         # 640
    detect_every_n = config.DETECT_EVERY_N_FRAMES,      # 3
)

print(f'Model type   : {detector.model_type}')
print(f'Model path   : {detector.model_path.name}')
print(f'Conf         : {detector.conf}')
print(f'Imgsz        : {detector.imgsz}')
print(f'Detect every : {detector.detect_every_n} frames')

## Cell 4 — Sample Action Frames

The first 25s of this clip is pre-match / empty pitch. We sample from 30–90s where football is happening.

In [ ]:
# 8 frames spread across the action section
sample_timestamps = [30, 40, 50, 60, 70, 75, 80, 90]
sample_indices    = [int(t * FPS) for t in sample_timestamps]
sample_frames     = [read_frame(idx) for idx in sample_indices]

print(f'Sampled {len(sample_frames)} frames: {sample_timestamps}s')

fig, axes = plt.subplots(2, 4, figsize=(20, 6))
fig.suptitle('Action-Section Sample Frames', fontsize=12, fontweight='bold')
for ax, frame, ts in zip(axes.flat, sample_frames, sample_timestamps):
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{ts}s', fontsize=9)
    ax.axis('off')
plt.tight_layout()
show(fig)

## Cell 5 — Run Detection on Sample Frames

Using `detect_raw()` here (no frame-index caching) so every sample frame goes through inference.

In [ ]:
sample_detections = []
print(f'Running detection on {len(sample_frames)} frames...')
print(f'{"ts":>4}s  {"persons":>7}  {"balls":>5}  {"total":>5}  ms')
print('-' * 35)

for frame, ts in zip(sample_frames, sample_timestamps):
    t0   = time.perf_counter()
    dets = detector.detect_raw(frame)
    ms   = (time.perf_counter() - t0) * 1000
    sample_detections.append(dets)
    p = sum(1 for d in dets if d.class_name in ('person', 'player', 'goalkeeper'))
    b = sum(1 for d in dets if d.class_name in ('ball', 'sports ball'))
    print(f'{ts:>4}s  {p:>7}  {b:>5}  {len(dets):>5}  {ms:.0f}ms')

total_players = sum(
    sum(1 for d in dets if d.class_name in ('person', 'player', 'goalkeeper'))
    for dets in sample_detections
)
print(f'\nTotal player detections across {len(sample_frames)} frames: {total_players}')
print(f'Mean per frame: {total_players / len(sample_frames):.1f}')

## Cell 6 — Visualise Raw Detections

Before team assignment — shows what the base COCO model sees.

In [ ]:
def draw_raw(frame, detections, thickness=2):
    ann = frame.copy()
    for det in detections:
        x1, y1, x2, y2 = det.bbox
        color = (50, 50, 220) if det.class_name in ('person','player','goalkeeper') else (0, 220, 220)
        cv2.rectangle(ann, (x1, y1), (x2, y2), color, thickness)
        label = f'{det.class_name} {det.confidence:.2f}'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
        cv2.rectangle(ann, (x1, y1-th-4), (x1+tw+2, y1), color, -1)
        cv2.putText(ann, label, (x1+1, y1-3), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1, cv2.LINE_AA)
    return ann

# Show 4 frames
show_idx = [1, 3, 5, 7]
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle(f'Raw Detections — conf={config.DETECTION_CONF_THRESHOLD}, imgsz={config.DETECTION_IMG_SIZE}',
             fontsize=11, fontweight='bold')
for ax, i in zip(axes, show_idx):
    ann = draw_raw(sample_frames[i], sample_detections[i])
    n   = len(sample_detections[i])
    ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{sample_timestamps[i]}s — {n} detections', fontsize=9)
    ax.axis('off')
plt.tight_layout()
show(fig)

## Cell 7 — Jersey Crop Extraction

**Why we crop, not use the full bounding box:**
- Head: skin colour pollutes the jersey signal
- Shorts: often a different colour to the shirt (e.g. white shirt, dark shorts)
- Legs: skin + socks would dominate if included

We take rows from `top + 10%` to `bottom - 20%` of the bounding box height.
This isolates the torso/shirt region.

**Why HSV not RGB:**
- Hue (H) separates colours independently of lighting brightness
- Saturation (S) lets us filter shadows (dark=low V) and near-white pixels
- RGB treats brightness and colour as mixed, making clustering noisier

In [ ]:
assigner = TeamAssigner(
    n_clusters  = config.TEAM_KMEANS_CLUSTERS,   # 3: teamA + teamB + referee
    crop_top    = config.JERSEY_CROP_TOP,         # skip top 10% (head)
    crop_bottom = config.JERSEY_CROP_BOTTOM,      # skip bottom 20% (shorts)
)

# Collect crops from the 60s frame for visualisation
ref_frame = sample_frames[3]   # 60s
ref_dets  = sample_detections[3]

player_dets  = [d for d in ref_dets if d.class_name in ('person','player','goalkeeper')]
jersey_crops = [assigner.get_jersey_crop(ref_frame, d.bbox) for d in player_dets]
jersey_crops = [c for c in jersey_crops if c is not None]

print(f'Players at 60s: {len(player_dets)}')
print(f'Valid jersey crops: {len(jersey_crops)}')

# Show crops in a grid
n_show  = min(len(jersey_crops), 16)
ncols   = 8
nrows   = (n_show + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.5, nrows * 2.5))
fig.suptitle(f'Jersey Crops (60s frame) — top={config.JERSEY_CROP_TOP:.0%} bottom={config.JERSEY_CROP_BOTTOM:.0%} excluded',
             fontsize=10, fontweight='bold')
axes = axes.flat if nrows > 1 else [axes] if ncols == 1 else axes.flat
for ax, crop in zip(axes, jersey_crops[:n_show]):
    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.axis('off')
for ax in list(axes)[n_show:]:
    ax.axis('off')
plt.tight_layout()
show(fig)

## Cell 8 — Fit TeamAssigner

**Why K-Means k=3 (not k=2):**
Referees wear a distinct colour (usually yellow or black). If we force k=2, referee crops
leak into one of the team clusters and pollute its centroid. With k=3 the referee forms
its own cluster. We then pick the **two largest** clusters as the two teams — the referee
cluster is smallest because there's only 1–4 referees vs 11+ players per team.

**Why fit on 8 frames (not 1):**
A single frame might catch players mid-run where the crop is blurry or occluded.
More samples → more robust cluster centres. 8 frames across 30–90s covers enough
positional variation without being slow.

In [ ]:
print(f'Fitting TeamAssigner on {len(sample_frames)} frames...')
assigner.fit(sample_frames, sample_detections)

print(f'  Fit on {assigner.n_fit_samples} jersey crops')
print(f'  Cluster → team mapping: {assigner.cluster_to_team}')
print()

# Show cluster centres as colour swatches
team_colors = assigner.team_colors_bgr
print('Team colour centroids (BGR → display as RGB):')
for team_id, bgr in team_colors.items():
    r, g, b = int(bgr[2]), int(bgr[1]), int(bgr[0])
    print(f'  Team {team_id}: RGB=({r}, {g}, {b})')

# Visualise swatches
fig, axes = plt.subplots(1, len(team_colors), figsize=(4 * len(team_colors), 2.5))
if len(team_colors) == 1: axes = [axes]
fig.suptitle('K-Means Cluster Centres (jersey colours)', fontsize=11, fontweight='bold')
for ax, (team_id, bgr) in zip(axes, team_colors.items()):
    swatch = np.ones((80, 120, 3), dtype=np.uint8) * bgr
    ax.imshow(cv2.cvtColor(swatch, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Team {team_id}', fontsize=11)
    ax.axis('off')
plt.tight_layout()
show(fig)

## Cell 9 — Team-Coloured Detection Boxes

Draw boxes coloured by team assignment: team 0 = red, team 1 = blue, unassigned = grey.

In [ ]:
TEAM_PALETTE_BGR = {
    0:  config.TEAM_A_COLOR_BGR,   # red
    1:  config.TEAM_B_COLOR_BGR,   # blue
    -1: (128, 128, 128),           # grey = unassigned / referee
}

def draw_teams(frame, detections, thickness=2):
    ann = frame.copy()
    for det in detections:
        x1, y1, x2, y2 = det.bbox
        color = TEAM_PALETTE_BGR.get(det.team_id, (128, 128, 128))
        cv2.rectangle(ann, (x1, y1), (x2, y2), color, thickness)

        if det.class_name in ('ball', 'sports ball'):
            label = 'ball'
        else:
            tid = det.team_id
            label = f'T{tid}' if tid >= 0 else 'ref'

        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(ann, (x1, y1-th-5), (x1+tw+3, y1), color, -1)
        cv2.putText(ann, label, (x1+1, y1-3), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)
    return ann

# Assign teams and visualise on 4 frames
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Team Assignment — Red=Team0  Blue=Team1  Grey=Referee/Unknown',
             fontsize=11, fontweight='bold')

for ax, i in zip(axes, show_idx):
    frame = sample_frames[i]
    dets  = sample_detections[i]
    assigned = assigner.assign(frame, dets)
    ann = draw_teams(frame, assigned)

    t0 = sum(1 for d in assigned if d.team_id == 0)
    t1 = sum(1 for d in assigned if d.team_id == 1)
    ref = sum(1 for d in assigned if d.team_id == -1 and d.class_name not in ('ball','sports ball'))

    ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{sample_timestamps[i]}s\nT0={t0}  T1={t1}  ref/unk={ref}', fontsize=9)
    ax.axis('off')

legend = [
    mpatches.Patch(color=(220/255,0,0),    label='Team 0'),
    mpatches.Patch(color=(0,0,220/255),    label='Team 1'),
    mpatches.Patch(color=(.5,.5,.5),       label='Ref / unknown'),
]
fig.legend(handles=legend, loc='lower center', ncol=3, fontsize=10)
plt.tight_layout(rect=[0, 0.06, 1, 1])
show(fig)

## Cell 10 — Per-Frame Team Count Table

Sanity check: both teams should have roughly equal player counts. Large imbalances indicate
the K-Means split is off (e.g. white kits blending with background).

In [ ]:
print(f'{"ts":>4}s  {"T0":>4}  {"T1":>4}  {"ref":>4}  {"total":>6}  balance')
print('-' * 40)

balances = []
for frame, dets, ts in zip(sample_frames, sample_detections, sample_timestamps):
    assigned = assigner.assign(frame, dets)
    t0  = sum(1 for d in assigned if d.team_id == 0)
    t1  = sum(1 for d in assigned if d.team_id == 1)
    ref = sum(1 for d in assigned if d.team_id == -1 and d.class_name not in ('ball','sports ball'))
    total = t0 + t1 + ref
    ratio = min(t0, t1) / max(t0, t1) if max(t0, t1) > 0 else 0
    balances.append(ratio)
    bar = '#' * int(ratio * 10)
    print(f'{ts:>4}s  {t0:>4}  {t1:>4}  {ref:>4}  {total:>6}  {ratio:.2f} {bar}')

print(f'\nMean T0/T1 balance: {np.mean(balances):.2f}  (1.0 = perfect)')
if np.mean(balances) >= 0.6:
    print('  -> GOOD — teams split reasonably evenly')
elif np.mean(balances) >= 0.4:
    print('  -> MODERATE — some confusion, likely white/light kit issue')
else:
    print('  -> POOR — one cluster dominates; try fitting on more frames or adjusting crop params')

## Cell 11 — 50-Frame Stability Test

Team assignment must be **temporally stable** — the same player should not flip between
team 0 and team 1 across consecutive frames. We measure this by looking at frame-to-frame
changes in the team 0 count.

We also time the full `detect → assign` loop to confirm the pipeline is fast enough.

In [ ]:
N_STABILITY = 50
START_FRAME = int(60 * FPS)   # start at 60s (guaranteed action)

cap = cv2.VideoCapture(str(CLIP))
cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)

stability_lats = []
t0_counts      = []
t1_counts      = []
det_counts     = []

for frame_idx in range(N_STABILITY):
    ret, frame = cap.read()
    if not ret: break

    t0 = time.perf_counter()
    dets     = detector.detect(frame, START_FRAME + frame_idx)  # uses skip cache
    assigned = assigner.assign(frame, dets)
    stability_lats.append((time.perf_counter() - t0) * 1000)

    t0c = sum(1 for d in assigned if d.team_id == 0)
    t1c = sum(1 for d in assigned if d.team_id == 1)
    t0_counts.append(t0c)
    t1_counts.append(t1c)
    det_counts.append(len(dets))

cap.release()

lats      = np.array(stability_lats)
t0_counts = np.array(t0_counts)
t1_counts = np.array(t1_counts)

print(f'50-frame stability test (starting at {START_FRAME/FPS:.0f}s):')
print(f'  Pipeline latency : mean={lats.mean():.0f}ms  median={np.median(lats):.0f}ms  '
      f'p95={np.percentile(lats,95):.0f}ms')
print(f'  Note: skip-3 means only 1-in-3 frames run inference; others return cache')
print()
print(f'  Team 0 count : mean={t0_counts.mean():.1f}  std={t0_counts.std():.1f}  '
      f'min={t0_counts.min()}  max={t0_counts.max()}')
print(f'  Team 1 count : mean={t1_counts.mean():.1f}  std={t1_counts.std():.1f}  '
      f'min={t1_counts.min()}  max={t1_counts.max()}')

t0_flips = np.sum(np.abs(np.diff(t0_counts)) > 2)
t1_flips = np.sum(np.abs(np.diff(t1_counts)) > 2)
print(f'\n  Team 0 large jumps (>2 count change frame-to-frame): {t0_flips}')
print(f'  Team 1 large jumps (>2 count change frame-to-frame): {t1_flips}')

if max(t0_flips, t1_flips) <= 5:
    print('  -> STABLE — team counts are consistent across frames')
else:
    print('  -> UNSTABLE — large swings suggest K-Means boundary is near the jersey colour')
    print('     Try: fit on more frames, adjust crop_top/crop_bottom, or use conf=0.30')

## Cell 12 — Stability Plot

In [ ]:
x = np.arange(N_STABILITY)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))

# Team counts
ax1.plot(x, t0_counts, color='#dc3545', linewidth=1.5, label='Team 0 (red)')
ax1.plot(x, t1_counts, color='#0047ab', linewidth=1.5, label='Team 1 (blue)')
ax1.fill_between(x, 0, t0_counts, alpha=0.10, color='#dc3545')
ax1.fill_between(x, 0, t1_counts, alpha=0.10, color='#0047ab')
ax1.axhline(t0_counts.mean(), color='#dc3545', linestyle='--', linewidth=0.8, alpha=0.5)
ax1.axhline(t1_counts.mean(), color='#0047ab', linestyle='--', linewidth=0.8, alpha=0.5)
ax1.set_ylabel('Players assigned')
ax1.set_title(f'Team Assignment Stability — {N_STABILITY} frames starting at {START_FRAME/FPS:.0f}s')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(bottom=0)

# Latency (shows skip-3 pattern: every 3rd frame is slow, others near-zero)
ax2.plot(x, lats, color='steelblue', linewidth=0.8, alpha=0.8)
ax2.axhline(lats.mean(), color='red', linestyle='--', linewidth=1.5,
            label=f'Mean {lats.mean():.0f}ms (cached frames near 0ms)')
ax2.set_xlabel('Frame #')
ax2.set_ylabel('detect+assign time (ms)')
ax2.set_title('Pipeline Latency (spike every 3 frames = inference run)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
show(fig)

## Cell 13 — Side-by-Side: Before vs After Team Assignment

In [ ]:
ref_frame2 = read_frame(int(75 * FPS))   # 75s — typically mid-match action
ref_dets2  = detector.detect_raw(ref_frame2)
assigned2  = assigner.assign(ref_frame2, ref_dets2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('75s Frame — Before vs After Team Assignment', fontsize=12, fontweight='bold')

ax1.imshow(cv2.cvtColor(draw_raw(ref_frame2, ref_dets2), cv2.COLOR_BGR2RGB))
ax1.set_title(f'Raw detections ({len(ref_dets2)} total)', fontsize=10)
ax1.axis('off')

ann2 = draw_teams(ref_frame2, assigned2)
t0c  = sum(1 for d in assigned2 if d.team_id == 0)
t1c  = sum(1 for d in assigned2 if d.team_id == 1)
refc = sum(1 for d in assigned2 if d.team_id == -1 and d.class_name not in ('ball','sports ball'))
ax2.imshow(cv2.cvtColor(ann2, cv2.COLOR_BGR2RGB))
ax2.set_title(f'Team assignment: T0={t0c}  T1={t1c}  ref/unk={refc}', fontsize=10)
ax2.axis('off')

plt.tight_layout()
show(fig)

## Cell 14 — Day 2 Summary

In [ ]:
print('=' * 55)
print('  DAY 2 SUMMARY')
print('=' * 55)
print()
print('  Files built:')
print('    gaffer/detection/detector.py     (FootballDetector)')
print('    gaffer/detection/team_assigner.py (TeamAssigner)')
print()
print('  What works:')
print(f'    FootballDetector wraps YOLO, skip-3 cache validated')
print(f'    TeamAssigner fits K-Means on {assigner.n_fit_samples} jersey crops')
print(f'    Team count balance: {np.mean(balances):.2f} avg ratio')
print(f'    Pipeline latency: {lats.mean():.0f}ms mean (most frames = cache hit ~0ms)')
print()
print('  Known limitations:')
print('    - No tracking: same player can get different team ID in adjacent frames')
print('      (this is what ByteTrack solves in Day 3)')
print('    - Ball detection still unreliable (base COCO model)')
print('    - Team colour assignment is based on jersey HSV mean — works well')
print('      for distinctly coloured kits but may struggle with similar colours')
print()
print('  Next:')
print('    Day 3 — ByteTrack: stable track IDs across frames')
print('    Day 4 — First annotated video output (v0.1)')
print('    Day 5 — Roboflow fine-tuning (goalkeeper/referee/ball classes)')
print('=' * 55)